# Gödel trace — Pythia forward pass, בלי TransformerLens

הנוטבוק הזה:
- לא מאמן שום מודל.
- טוען `EleutherAI/pythia-70m` ישירות דרך Hugging Face.
- מריץ forward pass אמיתי.
- בונה ייצוג גדל רקורסיבי של הפעולות והמקורות.
- לא מחשב את המספר הענק עצמו; שומר את המבנה שמגדיר אותו.


In [ ]:
!pip -q install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 5.6 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from dataclasses import dataclass, field
from typing import List, Dict, Any

MODEL_NAME = "EleutherAI/pythia-70m"
PROMPT = "The dog chased the red ball."

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# eager כדי שנוכל לבקש attention matrices גם בגרסאות חדשות של Transformers
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager",
).to(device)

model.eval()

inputs = tokenizer(PROMPT, return_tensors="pt").to(device)

with torch.no_grad():
    out = model(
        **inputs,
        output_attentions=True,
        output_hidden_states=True,
        use_cache=False,
        return_dict=True,
    )

tokens = inputs["input_ids"][0].tolist()
str_tokens = tokenizer.convert_ids_to_tokens(tokens)
next_id = int(out.logits[0, -1].argmax())

print("device:", device)
print("tokens:", list(enumerate(str_tokens)))
print("next-token prediction:", repr(tokenizer.decode([next_id])))
print("layers with attention:", len(out.attentions))
print("attention shape layer 0:", tuple(out.attentions[0].shape))
print("hidden-state tensors:", len(out.hidden_states))


config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  166MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

device: cpu
tokens: [(0, 'The'), (1, 'Ġdog'), (2, 'Ġchased'), (3, 'Ġthe'), (4, 'Ġred'), (5, 'Ġball'), (6, '.')]
next-token prediction: '<|endoftext|>'
layers with attention: 6
attention shape layer 0: (1, 8, 7, 7)
hidden-state tensors: 7


## קידוד גדל

אנחנו קובעים מראש את סוגי הפעולות. אין כאן שום classifier ושום למידה נוספת.

פורמלית, לצומת עם פעולה \(o\) והורים \(G_1,G_2,\ldots\):

\[
G_{	ext{new}}
=
2^{o}\,3^{G_1}\,5^{G_2}\,7^{G_3}\cdots
\]

לא נחשב את המספר הזה בפועל.

In [ ]:
OP = {
    "TOKEN": 1,
    "EMBED": 2,
    "LN": 3,
    "QKV": 4,
    "ATTN_SCORE": 5,
    "SOFTMAX": 6,
    "ATTEND": 7,
    "ATTN_OUT": 8,
    "MLP": 9,
    "ADD": 10,
    "FINAL_LN": 11,
    "UNEMBED": 12,
}

@dataclass
class GNode:
    idx: int
    op: str
    parents: List[int] = field(default_factory=list)
    meta: Dict[str, Any] = field(default_factory=dict)

nodes = []

def add_node(op, parents=None, **meta):
    node = GNode(
        idx=len(nodes),
        op=op,
        parents=list(parents or []),
        meta=meta,
    )
    nodes.append(node)
    return node.idx

def first_primes(n):
    primes = []
    x = 2
    while len(primes) < n:
        ok = True
        for p in primes:
            if p*p > x:
                break
            if x % p == 0:
                ok = False
                break
        if ok:
            primes.append(x)
        x += 1
    return primes

def formal_code(node_id, max_parents=5):
    n = nodes[node_id]
    primes = first_primes(1 + len(n.parents))
    parts = [f"2^{OP[n.op]}"]
    for j, parent in enumerate(n.parents[:max_parents], start=1):
        parts.append(f"{primes[j]}^G{parent}")
    if len(n.parents) > max_parents:
        parts.append(f"...(+{len(n.parents)-max_parents} parents)")
    return " * ".join(parts)


In [ ]:
# בניית trace מבני של ה-forward pass
#
# חשוב: זהו קידוד ברמת וקטורים/פעולות, לא כל כפל סקלרי בודד.
# ה-attention tensors שהתקבלו למעלה הם מה-forward pass האמיתי.

T = len(tokens)
L = len(out.attentions)
H = out.attentions[0].shape[1]

# טוקן -> embedding
resid = []
for pos, tok_id in enumerate(tokens):
    t = add_node(
        "TOKEN",
        [],
        token_id=tok_id,
        token_text=str_tokens[pos],
        position=pos,
    )
    e = add_node("EMBED", [t], position=pos)
    resid.append(e)

# לכל שכבה: LN -> QKV -> attention scores -> softmax -> weighted attend
# ואז attention output + MLP + residual add
for layer in range(L):
    new_resid = []

    # קודם בונים QKV לכל מיקום
    qkv = {}
    for pos in range(T):
        ln = add_node("LN", [resid[pos]], layer=layer, position=pos)
        for head in range(H):
            qkv[(head, pos)] = add_node(
                "QKV",
                [ln],
                layer=layer,
                head=head,
                position=pos,
            )

    attend_by_pos = {pos: [] for pos in range(T)}

    for head in range(H):
        for target in range(T):
            score_nodes = []
            for source in range(target + 1):  # causal
                # actual attention probability from the real forward pass
                attn_value = float(out.attentions[layer][0, head, target, source].cpu())

                score = add_node(
                    "ATTN_SCORE",
                    [qkv[(head, target)], qkv[(head, source)]],
                    layer=layer,
                    head=head,
                    target=target,
                    source=source,
                    attention=attn_value,
                )
                score_nodes.append(score)

            soft = add_node(
                "SOFTMAX",
                score_nodes,
                layer=layer,
                head=head,
                target=target,
            )

            # כל values האפשריים הם הורים; משקל ה-attention נשמר רק כ-metadata.
            value_parents = [qkv[(head, source)] for source in range(target + 1)]

            attend = add_node(
                "ATTEND",
                [soft] + value_parents,
                layer=layer,
                head=head,
                target=target,
            )
            attend_by_pos[target].append(attend)

    for pos in range(T):
        attn_out = add_node(
            "ATTN_OUT",
            attend_by_pos[pos],
            layer=layer,
            position=pos,
        )

        mlp = add_node(
            "MLP",
            [resid[pos]],
            layer=layer,
            position=pos,
        )

        merged = add_node(
            "ADD",
            [resid[pos], attn_out, mlp],
            layer=layer,
            position=pos,
        )
        new_resid.append(merged)

    resid = new_resid

final_ln = add_node("FINAL_LN", [resid[-1]], position=T-1)
final_node = add_node(
    "UNEMBED",
    [final_ln],
    predicted_token_id=next_id,
    predicted_token=tokenizer.decode([next_id]),
)

print("Gödel nodes created:", len(nodes))
print("final node:", f"G{final_node}")
print("top-level formal code:")
print("G_final =", formal_code(final_node))


Gödel nodes created: 2536
final node: G2535
top-level formal code:
G_final = 2^12 * 3^G2534


In [ ]:
# נפתח כמה שכבות לאחור כדי לראות שהקוד באמת רקורסיבי.

def show_trace(node_id, depth=0, max_depth=5):
    n = nodes[node_id]
    indent = "  " * depth

    meta = ""
    if n.op == "TOKEN":
        meta = f"  {n.meta['token_text']!r} @pos {n.meta['position']}"
    elif "layer" in n.meta:
        meta = f"  layer={n.meta['layer']}"
        if "head" in n.meta:
            meta += f", head={n.meta['head']}"

    print(f"{indent}G{node_id}: {n.op}{meta}")
    print(f"{indent}  {formal_code(node_id, max_parents=3)}")

    if depth >= max_depth:
        return

    for parent in n.parents[:6]:
        show_trace(parent, depth+1, max_depth)

show_trace(final_node)


G2535: UNEMBED
  2^12 * 3^G2534
  G2534: FINAL_LN
    2^11 * 3^G2533
    G2533: ADD  layer=5
      2^10 * 3^G2113 * 5^G2531 * 7^G2532
      G2113: ADD  layer=4
        2^10 * 3^G1693 * 5^G2111 * 7^G2112
        G1693: ADD  layer=3
          2^10 * 3^G1273 * 5^G1691 * 7^G1692
          G1273: ADD  layer=2
            2^10 * 3^G853 * 5^G1271 * 7^G1272
          G1691: ATTN_OUT  layer=3
            2^8 * 3^G1378 * 5^G1420 * 7^G1462 * ...(+5 parents)
          G1692: MLP  layer=3
            2^9 * 3^G1273
        G2111: ATTN_OUT  layer=4
          2^8 * 3^G1798 * 5^G1840 * 7^G1882 * ...(+5 parents)
          G1798: ATTEND  layer=4, head=0
            2^7 * 3^G1797 * 5^G1695 * 7^G1704 * ...(+5 parents)
          G1840: ATTEND  layer=4, head=1
            2^7 * 3^G1839 * 5^G1696 * 7^G1705 * ...(+5 parents)
          G1882: ATTEND  layer=4, head=2
            2^7 * 3^G1881 * 5^G1697 * 7^G1706 * ...(+5 parents)
          G1924: ATTEND  layer=4, head=3
            2^7 * 3^G1923 * 5^G1698 * 7^G1

In [ ]:
# sanity checks על ה-forward pass האמיתי
print("Attention row sums (צריכים להיות ~1):")
for layer in range(min(2, L)):
    row = out.attentions[layer][0, 0, -1]
    print("layer", layer, "sum =", float(row.sum().cpu()))

print("\nTop attention sources for final position:")
for layer in range(L):
    for head in range(H):
        row = out.attentions[layer][0, head, -1]
        source = int(row.argmax())
        value = float(row[source].cpu())
        print(
            f"L{layer} H{head}: source pos={source}, "
            f"token={str_tokens[source]!r}, attention={value:.4f}"
        )


Attention row sums (צריכים להיות ~1):
layer 0 sum = 1.0
layer 1 sum = 1.0

Top attention sources for final position:
L0 H0: source pos=6, token='.', attention=0.4275
L0 H1: source pos=6, token='.', attention=0.3237
L0 H2: source pos=0, token='The', attention=0.4944
L0 H3: source pos=6, token='.', attention=0.8052
L0 H4: source pos=6, token='.', attention=0.8906
L0 H5: source pos=3, token='Ġthe', attention=0.5493
L0 H6: source pos=0, token='The', attention=0.4348
L0 H7: source pos=0, token='The', attention=0.3403
L1 H0: source pos=6, token='.', attention=0.8384
L1 H1: source pos=6, token='.', attention=0.8779
L1 H2: source pos=6, token='.', attention=0.7017
L1 H3: source pos=6, token='.', attention=0.6001
L1 H4: source pos=6, token='.', attention=0.8237
L1 H5: source pos=6, token='.', attention=0.7910
L1 H6: source pos=6, token='.', attention=0.6382
L1 H7: source pos=6, token='.', attention=0.6772
L2 H0: source pos=6, token='.', attention=0.8291
L2 H1: source pos=6, token='.', attention

## מה השלב הבא?

אם הנוטבוק הזה רץ תקין, נריץ אותו על כמה prompts ונבדוק אילו תתי-קודים/גורמים חוזרים.

רק אז מתחילים `gcd`, חליקות והשוואה בין חישובים. אין צורך לאמן שום מודל.